# C11_E02 — PID discreto incremental con saturación

**Capítulo:** 11 — Control digital, muestreo e implementación en PLC/DCS  
**Identificador:** `C11_E02`  
**Objetivo pedagógico:** Implementar PID en estilo PLC con algoritmo incremental y saturación.  
**Librerías:** numpy, matplotlib

## Ejemplo industrial

PID en bloque de PLC con ciclo de scan determinístico.

---

> **Aviso de uso responsable.** Este notebook es didáctico. No es código de producción. Cualquier implementación real requiere validación independiente. Ver `docs/politica_uso_responsable.md`.

## 1. PID discreto incremental

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
np.random.seed(0)

class PIDIncremental:
    """PID en estilo PLC con saturación y antiwindup back-calculation."""
    def __init__(self, Kp, Ti, Td, Ts, umin=0, umax=100, Tt=1.0):
        self.Kp, self.Ti, self.Td, self.Ts = Kp, Ti, Td, Ts
        self.umin, self.umax, self.Tt = umin, umax, Tt
        self.e1 = self.e2 = 0.0
        self.u = 0.0
    def step(self, sp, pv):
        e = sp - pv
        du = self.Kp*((e - self.e1)
                      + (self.Ts/self.Ti)*e
                      + (self.Td/self.Ts)*(e - 2*self.e1 + self.e2))
        u_new = self.u + du
        u_sat = float(np.clip(u_new, self.umin, self.umax))
        self.u = u_sat
        self.e2, self.e1 = self.e1, e
        return u_sat

# Planta de primer orden discretizada explícitamente con Euler
def planta_step(y, u, Ts, K=2.0, tau=10.0):
    return y + Ts*(K*u - y)/tau

## 2. Simulación de un escalón en consigna

In [2]:
Ts = 0.2; T = 60.0
N = int(T/Ts); t = np.arange(N)*Ts
SP = 5.0; y = 0.0
pid = PIDIncremental(Kp=0.6, Ti=10.0, Td=0.0, Ts=Ts, umin=0, umax=4.0)
ys, us = [], []
for k in range(N):
    u = pid.step(SP, y)
    y = planta_step(y, u, Ts)
    ys.append(y); us.append(u)
ys, us = np.array(ys), np.array(us)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
ax1.plot(t, ys); ax1.axhline(SP, ls='--', color='gray')
ax1.set_ylabel("y"); ax1.grid(alpha=0.3)
ax1.set_title("C11_E02 — PID discreto incremental")
ax2.step(t, us, where='post'); ax2.set_xlabel("t (s)"); ax2.set_ylabel("u (saturada)")
ax2.grid(alpha=0.3)
out = '../../figures/cap11/fig_C11_F02_pid_discreto.png'
os.makedirs(os.path.dirname(out), exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=120); plt.show()

## 3. Interpretación

El PID incremental almacena solo `e1`, `e2` y `u`, lo que facilita la implementación en PLC y el bumpless transfer. La saturación del actuador se gestiona en `np.clip`. El comportamiento es equivalente al PID continuo cuando `Ts << τ`.